# Focos de calor en Argentina: armonización MODIS – VIIRS
**NASA Space Apps Challenge 2026 · Desafío CONAE**

Este notebook toma los datos de NASA FIRMS, **compara varios métodos de armonización con datos reales**,
elige el mejor según los resultados y guarda los archivos que usa la página web.

| Paso | Qué hace |
|---|---|
| 1 | Conecta Google Drive y define las regiones |
| 2 | Lee los zips de FIRMS (MODIS y VIIRS) |
| 3 | Limpia los datos y los resume por región y mes, con **3 formas de medir** |
| 4 | Muestra el problema: el salto falso de 2012 |
| 5 | **Compara 4 métodos de armonización** (entrenamiento 2012–2021, test 2022–2025) |
| 6 | **Elige el método** con una regla clara y lo justifica con los números |
| 7 | Valida: ¿las 3 formas de medir cuentan la misma historia? |
| 8 | Arma la serie armonizada 2003–2026 y guarda todo en Drive |

👉 Menú **Entorno de ejecución → Ejecutar todas**. Tarda unos 5 a 10 minutos.

## 1. Configuración

Los zips de FIRMS tienen que estar en Drive, en **Mi unidad / Challenge NASA 2026 / datos**, tal como vienen (sin descomprimir).
Al ejecutar, Colab pide permiso para entrar a Drive: hay que aceptar.

- **MODIS desde 2003:** desde ese año funcionan Terra y Aqua todo el año, así la serie es pareja.
- **VIIRS solo de Suomi NPP:** es el único VIIRS que funciona desde 2012.

In [ ]:
import os, json, glob, zipfile, shutil, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

# ---- Carpeta del proyecto en Google Drive (dentro de "Mi unidad") ----
CARPETA_DRIVE = "Challenge NASA 2026"
SUBCARPETA_DATOS = "datos"        # donde están los .zip de FIRMS
SUBCARPETA_SALIDA = "salida"      # acá se guardan los resultados para la página web

CARPETA_DATOS = "datos_firms"     # carpeta de trabajo local de Colab
CARPETA_SALIDA = "salida"
os.makedirs(CARPETA_DATOS, exist_ok=True)
os.makedirs(CARPETA_SALIDA, exist_ok=True)

EN_COLAB = False
try:
    from google.colab import drive
    drive.mount("/content/drive")
    EN_COLAB = True
except ImportError:
    print("No estamos en Colab: se usan los archivos de la carpeta local", CARPETA_DATOS)

RUTA_DRIVE = f"/content/drive/MyDrive/{CARPETA_DRIVE}"
if EN_COLAB:
    origen = os.path.join(RUTA_DRIVE, SUBCARPETA_DATOS)
    if not os.path.isdir(origen):
        raise FileNotFoundError(f"No encuentro 'Mi unidad/{CARPETA_DRIVE}/{SUBCARPETA_DATOS}'. Revisen el nombre.")
    for f in os.listdir(origen):
        if f.lower().endswith((".zip", ".csv")) and not os.path.exists(os.path.join(CARPETA_DATOS, f)):
            shutil.copy(os.path.join(origen, f), CARPETA_DATOS)
            print("Copiado desde Drive:", f)

ANIO_INICIO_MODIS = 2003
ANIO_INICIO_VIIRS = 2012
ANIOS_ENTRENAMIENTO = list(range(2012, 2022))   # 2012–2021
ANIOS_TEST = list(range(2022, 2026))            # 2022–2025 (años completos que el modelo no ve)
TAMANIO_CELDA = 0.25                            # grados (~25 km)

# Regiones: nombre -> (lat_min, lat_max, lon_min, lon_max). Se pueden editar o agregar.
REGIONES = {
    "Gran Chaco":         (-29.5, -22.0, -65.5, -59.6),
    "Delta del Paraná":   (-34.4, -32.0, -61.0, -58.3),
    "Sierras de Córdoba": (-33.5, -30.0, -65.5, -64.0),
    "Corrientes e Iberá": (-30.8, -27.2, -59.5, -55.6),
    "Patagonia andina":   (-46.0, -39.0, -72.5, -70.5),
}
RESTO = "Resto del país"
PAIS = "Argentina (todo el país)"
REGIONES_MODELO = list(REGIONES) + [RESTO]

def region_de(lat, lon):
    "Región de cada punto (se usa el centro de la celda, así celda y región siempre coinciden)."
    r = pd.Series(RESTO, index=lat.index)
    for nombre, (la0, la1, lo0, lo1) in REGIONES.items():
        r[lat.between(la0, la1) & lon.between(lo0, lo1)] = nombre
    return r

## 2. Lectura de los archivos

Cada zip de FIRMS trae un archivo **definitivo** (`fire_archive_...`) y uno de **tiempo casi real** (`fire_nrt_...`).
El sensor se reconoce por la columna `instrument` (`MODIS` o `SNPP`/`VIIRS`).
Los archivos `nrt` empiezan justo donde terminan los definitivos, así que ningún foco se cuenta dos veces.

In [ ]:
for z in glob.glob(os.path.join(CARPETA_DATOS, "*.zip")):
    with zipfile.ZipFile(z) as zf:
        zf.extractall(CARPETA_DATOS)

filas = []
for ruta in sorted(glob.glob(os.path.join(CARPETA_DATOS, "**", "*.csv"), recursive=True)):
    cab = pd.read_csv(ruta, nrows=5)
    if "instrument" not in cab.columns:
        continue
    instrumento = str(cab["instrument"].iloc[0]).upper()
    sensor = "modis" if "MODIS" in instrumento else "viirs"
    fechas = pd.to_datetime(pd.read_csv(ruta, usecols=["acq_date"])["acq_date"])
    filas.append({"archivo": os.path.basename(ruta), "ruta": ruta, "sensor": sensor,
                  "nrt": "nrt" in os.path.basename(ruta).lower(),
                  "desde": fechas.min().date(), "hasta": fechas.max().date(), "focos": len(fechas),
                  "MB": round(os.path.getsize(ruta) / 1e6, 1)})

inventario = pd.DataFrame(filas).sort_values(["nrt", "sensor"])   # definitivos primero
assert set(inventario.sensor) == {"modis", "viirs"}, "Faltan datos de MODIS o de VIIRS."
FECHA_FIN = pd.Timestamp(min(inventario.groupby("sensor")["hasta"].max()))
ANIO_FIN = FECHA_FIN.year
print(f"Datos hasta el {FECHA_FIN:%d/%m/%Y}")
inventario.drop(columns="ruta")

## 3. Limpieza y resumen

**Filtros (iguales para los dos sensores en lo posible):**
- Solo incendios de vegetación (`type = 0`). Se sacan las antorchas de gas, industrias y volcanes.
  Los archivos `nrt` no traen `type`: ahí se descartan los focos que caen en lugares donde
  el archivo definitivo ya registró fuentes fijas (a ~1 km).
- Confianza: MODIS ≥ 30 (de 0 a 100); VIIRS sin las detecciones de confianza baja (`l`).

**Tres formas de medir la actividad de fuego, por región y por mes:**

| Medida | Qué cuenta | Por qué |
|---|---|---|
| **Focos** | Cantidad de detecciones | Es la medida habitual, pero VIIRS divide cada incendio en más píxeles |
| **Días con fuego** | Pares (celda de 25 km, día) con al menos un foco | Un incendio vale lo mismo aunque tenga 1 o 5 píxeles |
| **Energía (FRP)** | Suma de la potencia radiativa, en MW | Mide energía, no cantidad de píxeles |

Los archivos se leen en partes de 500.000 filas para no llenar la memoria.

In [ ]:
COLS = ["latitude", "longitude", "acq_date", "acq_time", "satellite", "confidence", "frp", "type"]

def clave_1km(lat, lon):
    "Identificador de un punto redondeado a 0,01° (~1 km)."
    return np.round(lat * 100).astype("int64") * 100000 + np.round(lon * 100).astype("int64")

fuentes_fijas = {"modis": set(), "viirs": set()}
partes = {"conteo": [], "frp": [], "celdadia": [], "grilla": [], "aqua": [], "puntos": []}

for _, a in inventario.iterrows():
    n_leidas, n_usadas = 0, 0
    for df in pd.read_csv(a.ruta, usecols=lambda c: c in COLS, chunksize=500_000, low_memory=False):
        n_leidas += len(df)
        if "type" in df.columns:
            fijas = df[df["type"] != 0]
            fuentes_fijas[a.sensor].update(clave_1km(fijas.latitude, fijas.longitude).unique())
            df = df[df["type"] == 0]
        else:
            df = df[~clave_1km(df.latitude, df.longitude).isin(fuentes_fijas[a.sensor])]
        if a.sensor == "modis":
            df = df[pd.to_numeric(df["confidence"], errors="coerce") >= 30]
        else:
            df = df[df["confidence"].astype(str).str.lower() != "l"]
        fecha = pd.to_datetime(df["acq_date"])
        inicio = ANIO_INICIO_MODIS if a.sensor == "modis" else ANIO_INICIO_VIIRS
        df = df[(fecha.dt.year >= inicio) & (fecha <= FECHA_FIN)]
        if df.empty:
            continue
        fecha = pd.to_datetime(df["acq_date"])
        lat_c = (np.floor(df.latitude / TAMANIO_CELDA) * TAMANIO_CELDA + TAMANIO_CELDA / 2).round(3)
        lon_c = (np.floor(df.longitude / TAMANIO_CELDA) * TAMANIO_CELDA + TAMANIO_CELDA / 2).round(3)
        df = df.assign(anio=fecha.dt.year, mes=fecha.dt.month, lat_c=lat_c, lon_c=lon_c,
                       region=region_de(lat_c, lon_c), sensor=a.sensor)
        k = ["region", "anio", "mes", "sensor"]
        partes["conteo"].append(df.groupby(k).size().rename("valor").reset_index())
        partes["frp"].append(df.groupby(k)["frp"].sum().rename("valor").reset_index())
        partes["celdadia"].append(df[["lat_c", "lon_c", "acq_date", "region", "anio", "mes", "sensor"]].drop_duplicates())
        partes["grilla"].append(df.groupby(["lat_c", "lon_c", "region", "anio", "mes", "sensor"]).size()
                                .rename("focos").reset_index())
        if a.sensor == "modis":
            partes["aqua"].append(df.assign(sat=df.satellite.astype(str).str[0])
                                  .groupby(["anio", "sat"]).size().rename("n").reset_index())
        # Focos individuales (para buscar por fecha y por área en la página, y descargarlos)
        partes["puntos"].append(pd.DataFrame({
            "fecha": fecha.values.astype("datetime64[D]"),
            "hora_utc": pd.to_numeric(df["acq_time"], errors="coerce").fillna(0).astype("int16").values,
            "lat": df.latitude.astype("float32").values, "lon": df.longitude.astype("float32").values,
            "sensor": "MODIS" if a.sensor == "modis" else "VIIRS",
            "satelite": df.satellite.astype(str).values, "confianza": df.confidence.astype(str).values,
            "frp": df.frp.astype("float32").values, "region": region_de(df.latitude, df.longitude).values}))
        n_usadas += len(df)
    print(f"{a.archivo:34s} {n_leidas:>10,} leídos  {n_usadas:>10,} usados")

def a_tabla(df):
    return df.groupby(["region", "anio", "mes", "sensor"], as_index=False)["valor"].sum()

celdadia = pd.concat(partes["celdadia"]).drop_duplicates(["lat_c", "lon_c", "acq_date", "sensor"])
medidas = {
    "focos": a_tabla(pd.concat(partes["conteo"])),
    "dias":  a_tabla(celdadia.groupby(["region", "anio", "mes", "sensor"]).size().rename("valor").reset_index()),
    "frp":   a_tabla(pd.concat(partes["frp"])),
}
NOMBRE_MEDIDA = {"focos": "Focos", "dias": "Días con fuego", "frp": "Energía (FRP, MW)"}
grilla = pd.concat(partes["grilla"]).groupby(["lat_c", "lon_c", "region", "anio", "mes", "sensor"],
                                              as_index=False)["focos"].sum()
terra_aqua = pd.concat(partes["aqua"]).groupby(["anio", "sat"]).n.sum().unstack()

# Tabla completa: región × año × mes, con MODIS y VIIRS en columnas.
indice = pd.MultiIndex.from_product([REGIONES_MODELO, range(ANIO_INICIO_MODIS, ANIO_FIN + 1), range(1, 13)],
                                    names=["region", "anio", "mes"])
fin_mes = FECHA_FIN.year * 12 + FECHA_FIN.month

def armar(df):
    t = (df.pivot_table(index=["region", "anio", "mes"], columns="sensor", values="valor", aggfunc="sum")
         .reindex(indice).reset_index())
    t["modis"] = t["modis"].fillna(0)
    t["viirs"] = np.where(t.anio >= ANIO_INICIO_VIIRS, t["viirs"].fillna(0), np.nan)
    futuro = t.anio * 12 + t.mes > fin_mes          # meses que todavía no pasaron
    t.loc[futuro, ["modis", "viirs"]] = np.nan
    return t

tablas = {m: armar(df) for m, df in medidas.items()}
tablas["focos"].head()

## 4. El problema: el salto falso de 2012

Si se pegan MODIS (hasta 2011) y VIIRS (desde 2012), parece que el fuego se multiplicó en 2012.
Es el cambio de sensor, no un cambio real.

In [ ]:
def anual_pais(t, col):
    return t[t.anio <= 2025].groupby("anio")[col].sum(min_count=1)

fig, ejes = plt.subplots(1, 3, figsize=(16, 4))
for ax, m in zip(ejes, tablas):
    t = tablas[m]
    ax.plot(anual_pais(t, "modis"), "o-", label="MODIS")
    ax.plot(anual_pais(t, "viirs"), "s-", label="VIIRS")
    ax.axvline(2011.5, color="gray", lw=.8)
    ax.set_title(NOMBRE_MEDIDA[m]); ax.legend()
plt.suptitle("Argentina por año: MODIS y VIIRS miden distinto"); plt.tight_layout(); plt.show()

resumen = []
for m, t in tablas.items():
    c = t[t.anio.between(2012, 2025)].groupby("anio")[["modis", "viirs"]].sum()
    r = c.viirs / c.modis
    resumen.append({"medida": NOMBRE_MEDIDA[m], "VIIRS/MODIS promedio": r.mean(),
                    "mínimo anual": r.min(), "máximo anual": r.max(), "variación (CV %)": 100 * r.std() / r.mean()})
pd.DataFrame(resumen).set_index("medida")

## 5. Comparación de métodos de armonización

**Objetivo:** estimar cuánto habría medido VIIRS a partir de lo que midió MODIS, en cada región y mes.

**Validación honesta:** cada método aprende con **2012–2021** y se evalúa con **2022–2025**, años que no vio.

| # | Método | Fórmula / idea | Complejidad |
|---|---|---|---|
| 1 | Factor fijo por región | V = k · M | Muy baja |
| 2 | Regresión logarítmica por región y mes | log(1+V) = a + b·log(1+M) + c·sen(mes) + d·cos(mes) | Baja |
| 3 | Mapeo de cuantiles por región | V = F_V⁻¹(F_M(M)): se emparejan percentiles | Media |
| 4 | Random Forest | Promedio de 300 árboles de decisión | Alta |

En los métodos 2 y 4 se corrige el **sesgo de volver de la escala logarítmica** (desigualdad de Jensen):
se multiplica por un factor κ para que el total de entrenamiento coincida con el real.

**Métricas (sobre el período de test):**
- **MAE:** error promedio por región y mes. Más bajo es mejor.
- **R² (log):** qué parte de la variación explica (1 = perfecto).
- **Sesgo total:** cuánto se pasa (+) o se queda corto (−) en el total de 2022–2025.
- **Peor año (país):** el mayor error en el total anual de Argentina. Es lo que más importa para la página.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score

def estacion(df):
    return np.c_[np.sin(2 * np.pi * df.mes / 12), np.cos(2 * np.pi * df.mes / 12)]

# ---------- 1. Factor fijo por región
def ajustar_factor(tr):
    s = tr.groupby("region")[["modis", "viirs"]].sum()
    return (s.viirs / s.modis.replace(0, np.nan)).fillna(1.0).to_dict()

def predecir_factor(k, df):
    return df.modis * df.region.map(k)

# ---------- 2. Regresión logarítmica por región y mes (con corrección de sesgo)
def ajustar_regresion(tr):
    modelos = {}
    for r, sub in tr.groupby("region"):
        X = np.c_[np.log1p(sub.modis), estacion(sub)]
        lr = LinearRegression().fit(X, np.log1p(sub.viirs))
        kappa = sub.viirs.sum() / np.expm1(lr.predict(X)).clip(0).sum()
        modelos[r] = (lr, kappa)
    return modelos

def predecir_regresion(modelos, df):
    p = pd.Series(np.nan, index=df.index)
    for r, sub in df.groupby("region"):
        lr, kappa = modelos[r]
        p[sub.index] = np.expm1(lr.predict(np.c_[np.log1p(sub.modis), estacion(sub)])).clip(0) * kappa
    return p

# ---------- 3. Mapeo de cuantiles por región
def ajustar_cuantiles(tr):
    return {r: (np.sort(sub.modis.values), np.sort(sub.viirs.values),
                sub.viirs.sum() / max(sub.modis.sum(), 1))
            for r, sub in tr.groupby("region")}

def predecir_cuantiles(q, df):
    p = pd.Series(np.nan, index=df.index)
    for r, sub in df.groupby("region"):
        m_ord, v_ord, k = q[r]
        x = sub.modis.values
        # F_M(x): percentil de x entre los meses de entrenamiento (con empates, el percentil medio)
        prob = (np.searchsorted(m_ord, x, "left") + np.searchsorted(m_ord, x, "right")) / (2 * len(m_ord))
        # F_V^-1: valor de VIIRS en ese mismo percentil
        y = np.quantile(v_ord, np.clip(prob, 0, 1))
        mas_alto = x > m_ord[-1]                        # más alto que todo lo visto: se extrapola con el factor
        y[mas_alto] = v_ord[-1] + (x[mas_alto] - m_ord[-1]) * k
        p[sub.index] = y
    return p

# ---------- 4. Random Forest (con corrección de sesgo usando predicciones "out of bag")
def X_rf(df):
    X = pd.DataFrame({"log_modis": np.log1p(df.modis)}, index=df.index)
    X[["mes_sen", "mes_cos"]] = estacion(df)
    for r in REGIONES_MODELO:
        X["reg_" + r] = (df.region == r).astype(int)
    return X

def ajustar_rf(tr):
    rf = RandomForestRegressor(n_estimators=300, min_samples_leaf=3, oob_score=True,
                               random_state=42, n_jobs=-1).fit(X_rf(tr), np.log1p(tr.viirs))
    kappa = tr.viirs.sum() / np.expm1(rf.oob_prediction_).clip(0).sum()
    return rf, kappa

def predecir_rf(m, df):
    rf, kappa = m
    return pd.Series(np.expm1(rf.predict(X_rf(df))).clip(0) * kappa, index=df.index)

METODOS = {   # en orden de complejidad: importa para la regla de elección
    "Factor fijo por región":   (ajustar_factor, predecir_factor),
    "Regresión log por región y mes": (ajustar_regresion, predecir_regresion),
    "Mapeo de cuantiles":       (ajustar_cuantiles, predecir_cuantiles),
    "Random Forest":            (ajustar_rf, predecir_rf),
}

def evaluar(t):
    comun = t[t.anio.between(2012, 2025) & t.viirs.notna()]
    tr, te = comun[comun.anio.isin(ANIOS_ENTRENAMIENTO)], comun[comun.anio.isin(ANIOS_TEST)]
    filas, anual = [], {"Real": te.groupby("anio").viirs.sum()}
    for nombre, (ajustar, predecir) in METODOS.items():
        p = predecir(ajustar(tr), te)
        err_anual = 100 * (p.groupby(te.anio).sum() / te.groupby("anio").viirs.sum() - 1)
        filas.append({"método": nombre,
                      "MAE": (te.viirs - p).abs().mean(),
                      "R² (log)": r2_score(np.log1p(te.viirs), np.log1p(p)),
                      "Sesgo total %": 100 * (p.sum() / te.viirs.sum() - 1),
                      "Peor año (país) %": err_anual.abs().max()})
        anual[nombre] = p.groupby(te.anio).sum()
    return pd.DataFrame(filas).set_index("método"), pd.DataFrame(anual)

resultados, anual_test = {}, {}
for m, t in tablas.items():
    resultados[m], anual_test[m] = evaluar(t)
    print(f"\n=== {NOMBRE_MEDIDA[m]} ===")
    display(resultados[m].round(3))

In [ ]:
fig, ejes = plt.subplots(1, 3, figsize=(16, 4))
for ax, m in zip(ejes, tablas):
    d = anual_test[m]
    ax.plot(d.index, d["Real"], "k-o", lw=3, label="VIIRS real")
    for c in d.columns[1:]:
        ax.plot(d.index, d[c], "--", marker=".", label=c)
    ax.set_title(f"Test 2022–2025 · {NOMBRE_MEDIDA[m]}"); ax.set_xticks(d.index)
ejes[0].legend(fontsize=8); plt.tight_layout(); plt.show()

## 6. Elección del método

**Regla:** se elige el método **más simple** cuyo error (MAE) no supere en más de un **5 %** al mejor.
Así un modelo complejo solo gana si mejora de verdad; si empatan, gana el que se puede explicar.
La elección se hace por separado para cada medida y la justificación sale de la tabla de arriba.

In [ ]:
TOLERANCIA = 0.05
elegido, justificacion = {}, {}
for m, res in resultados.items():
    mejor_mae = res["MAE"].min()
    mejor = res["MAE"].idxmin()
    candidatos = res[res["MAE"] <= mejor_mae * (1 + TOLERANCIA)]
    elegido[m] = candidatos.index[0]          # el primero = el más simple (METODOS está ordenado)
    e = res.loc[elegido[m]]
    texto = (f"Se eligió «{elegido[m]}»: MAE {e['MAE']:,.1f} por región y mes, "
             f"sesgo total {e['Sesgo total %']:+.1f} % y peor año {e['Peor año (país) %']:.1f} % en 2022–2025. ")
    if elegido[m] == mejor:
        texto += "Es el de menor error."
    else:
        texto += (f"El de menor error es «{mejor}» (MAE {mejor_mae:,.1f}), pero la diferencia es de solo "
                  f"{100 * (e['MAE'] / mejor_mae - 1):.1f} %: se prefiere el más simple y explicable.")
    rf = res.loc["Random Forest", "MAE"]
    texto += f" El Random Forest tuvo MAE {rf:,.1f} ({100 * (rf / mejor_mae - 1):+.1f} % respecto del mejor)."
    justificacion[m] = texto
    print(f"{NOMBRE_MEDIDA[m]}:\n  {texto}\n")

## 7. Validación: ¿las tres medidas cuentan la misma historia?

Si la armonización es buena, los **años excepcionales** tienen que aparecer igual sin importar cómo se mida.
Se arma la serie armonizada de cada medida con su método elegido (entrenado con 2012–2025)
y se compara cada año contra su propio promedio (100 = año promedio).

In [ ]:
final = {}
for m, t in tablas.items():
    comun = t[t.anio.between(2012, 2025) & t.viirs.notna()]
    ajustar, predecir = METODOS[elegido[m]]
    modelo = ajustar(comun)
    t = t.copy()
    t["estimado"] = np.nan
    con_dato = t.modis.notna()
    t.loc[con_dato, "estimado"] = predecir(modelo, t[con_dato])
    t["equiv"] = np.where(t.anio >= ANIO_INICIO_VIIRS, t.viirs, t.estimado)
    final[m] = t

indice_anual = pd.DataFrame({NOMBRE_MEDIDA[m]: t[t.anio <= 2025].groupby("anio").equiv.sum() for m, t in final.items()})
indice_anual = 100 * indice_anual / indice_anual.mean()

ax = indice_anual.plot(figsize=(12, 4), marker="o")
ax.axhline(100, color="gray", lw=.8); ax.axvline(2011.5, color="gray", ls=":", lw=.8)
ax.set_title("Serie armonizada de Argentina (100 = año promedio)"); plt.show()

print("Correlación entre las tres medidas (años 2003–2025):")
display(indice_anual.corr().round(3))
print("Los 5 años más activos según cada medida:")
display(pd.DataFrame({c: indice_anual[c].nlargest(5).index.values for c in indice_anual}))

## 8. Serie final y archivos para la página web

Se guardan en Drive, en **Challenge NASA 2026 / salida**:
- `serie_armonizada.csv`: región × año × mes, con las tres medidas (MODIS, VIIRS y armonizado)
- `grilla_mensual.csv`: celdas de ~25 km para el mapa (focos armonizados con el factor de cada región)
- `regiones.json`, `metricas.json`: regiones, comparación de métodos, justificación y validación
- carpeta `focos/`: los focos individuales, un archivo por año, para buscar por fecha y por área en la página

Esos archivos se copian a la carpeta `datos/` de la página web.

In [ ]:
serie = None
for m, t in final.items():
    parte = t[["region", "anio", "mes", "modis", "viirs", "equiv"]].rename(
        columns={"modis": f"{m}_modis", "viirs": f"{m}_viirs", "equiv": f"{m}_equiv"})
    serie = parte if serie is None else serie.merge(parte, on=["region", "anio", "mes"])
pais = serie.groupby(["anio", "mes"]).sum(min_count=1).reset_index().assign(region=PAIS)
serie = pd.concat([pais, serie], ignore_index=True)
serie["fuente"] = np.where(serie.anio >= ANIO_INICIO_VIIRS, "VIIRS", "MODIS armonizado")
serie = serie.round(2)

# Mapa: focos por celda, convertidos con el factor de la región
factores = ajustar_factor(tablas["focos"][tablas["focos"].anio.between(2012, 2025)])
g = grilla.pivot_table(index=["lat_c", "lon_c", "region", "anio", "mes"], columns="sensor",
                       values="focos", fill_value=0).reset_index()
g["focos_equiv"] = np.where(g.anio >= ANIO_INICIO_VIIRS, g.viirs, g.modis * g.region.map(factores))
grilla_final = g[g.focos_equiv > 0][["lat_c", "lon_c", "region", "anio", "mes", "focos_equiv"]].round(2)

ta = (terra_aqua["T"] / terra_aqua["A"]).round(3) if {"T", "A"} <= set(terra_aqua.columns) else pd.Series(dtype=float)
metricas = {
    "fecha_fin": f"{FECHA_FIN:%Y-%m-%d}",
    "anios_entrenamiento": f"{ANIOS_ENTRENAMIENTO[0]}–{ANIOS_ENTRENAMIENTO[-1]}",
    "anios_test": f"{ANIOS_TEST[0]}–{ANIOS_TEST[-1]}",
    "tolerancia": TOLERANCIA,
    "nombres": NOMBRE_MEDIDA,
    "elegido": elegido,
    "justificacion": justificacion,
    "resultados": {m: r.reset_index().to_dict(orient="records") for m, r in resultados.items()},
    "anual_test": {m: d.reset_index().rename(columns={"index": "anio"}).to_dict(orient="list") for m, d in anual_test.items()},
    "factores_focos": {k: round(v, 3) for k, v in factores.items()},
    "indice_anual": indice_anual.round(1).reset_index().to_dict(orient="list"),
    "correlacion_medidas": indice_anual.corr().round(3).to_dict(),
    "terra_sobre_aqua": {int(k): v for k, v in ta.items()},
    "factor_pais_focos": round(float(tablas["focos"].query("anio >= 2012 and anio <= 2025").viirs.sum()
                                     / tablas["focos"].query("anio >= 2012 and anio <= 2025").modis.sum()), 3),
}

serie.to_csv(os.path.join(CARPETA_SALIDA, "serie_armonizada.csv"), index=False)
grilla_final.to_csv(os.path.join(CARPETA_SALIDA, "grilla_mensual.csv"), index=False)

# Focos individuales: un archivo liviano por año (formato parquet), en la subcarpeta "focos"
puntos = pd.concat(partes["puntos"], ignore_index=True)
for c in ["sensor", "satelite", "confianza", "region"]:
    puntos[c] = puntos[c].astype("category")
os.makedirs(os.path.join(CARPETA_SALIDA, "focos"), exist_ok=True)
for a_, sub in puntos.groupby(puntos.fecha.dt.year):
    sub.sort_values(["fecha", "hora_utc"]).to_parquet(
        os.path.join(CARPETA_SALIDA, "focos", f"focos_{a_}.parquet"), index=False, compression="zstd")
print(f"Focos individuales guardados: {len(puntos):,} en {puntos.fecha.dt.year.nunique()} archivos")
with open(os.path.join(CARPETA_SALIDA, "regiones.json"), "w", encoding="utf-8") as f:
    json.dump(REGIONES, f, ensure_ascii=False, indent=2)
with open(os.path.join(CARPETA_SALIDA, "metricas.json"), "w", encoding="utf-8") as f:
    json.dump(metricas, f, ensure_ascii=False, indent=2, default=float)

for f in sorted(os.listdir(CARPETA_SALIDA)):
    ruta = os.path.join(CARPETA_SALIDA, f)
    tam = (sum(os.path.getsize(os.path.join(ruta, x)) for x in os.listdir(ruta)) if os.path.isdir(ruta)
           else os.path.getsize(ruta))
    print(f"{f:25s} {tam / 1e6:6.2f} MB")

if EN_COLAB:
    destino = os.path.join(RUTA_DRIVE, SUBCARPETA_SALIDA)
    os.makedirs(destino, exist_ok=True)
    shutil.copytree(CARPETA_SALIDA, destino, dirs_exist_ok=True)
    print(f"\nGuardado en Drive: Mi unidad/{CARPETA_DRIVE}/{SUBCARPETA_SALIDA}")